# Lab 05 — Reasoning Models & Agent Design

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- explain what distinguishes a *reasoning model* (RL-trained extended chain of thought) from a standard instruction-tuned LLM,
- run both model classes locally via Ollama and parse their **thinking traces**,
- measure **accuracy vs token cost vs latency** for both classes on a small task suite,
- plot **accuracy against a thinking budget** and locate the knee of the curve,
- rebuild the lecture's **minimal model router** and extend it with a **per-run thinking-budget cap**,
- decide per agent role (planner vs formatter) **which model class** a step should use.

> ⏱️ Estimated time: 90–120 minutes. Reasoning-model calls are slow (tens of seconds each on
> CPU or a small GPU) — start the long-running cells early and read on while they run.

## Theory recap — from prompted to trained reasoning

### What a reasoning model is

A **reasoning model** is an LLM trained — typically with **reinforcement learning (RL) on
verifiable outcomes** — to emit a long private **chain of thought** before committing to an
answer. These **thinking tokens** are generated autoregressively like any output and are
**billed like output**, even though interfaces hide or summarise them. Crucially, the behaviour
is **learned, not prompted**: in Session 04 we *elicited* reasoning with CoT prompts ("think
step by step", few-shot worked examples); a reasoning model brings its own. The reward signal
during training is *verifiable* — unit tests pass, the math answer matches. Under pure
outcome-based RL (the recipe published openly with DeepSeek-R1), traces grow longer on their
own, and behaviours **emerge** that nobody demonstrated: **self-verification**,
**backtracking**, **decomposition**. *Faithfulness caveat:* the visible trace is not a reliable
account of the computation (Anthropic, 2025) — use traces for debugging hypotheses, never as
evidence of correctness.

### Inference-time scaling: the third axis

Accuracy now scales with *how long the model thinks* — not only with how big it is. Next to
(1) pretraining compute and (2) post-training, **test-time compute** is a third dial, set per
request. On hard math, code and planning, accuracy grows roughly **log-linearly** with thinking
tokens: each accuracy increment costs a constant *multiple* of tokens. The gains are sharply
task-dependent — large on tasks with deep dependency chains, **near zero on recall, extraction
and formatting**; budgets beyond task difficulty buy verbosity, not correctness. Two ways to
spend the compute: **serial** (one long chain — handles dependent steps, latency grows with the
budget) and **parallel** (sample $N$ answers and select — majority vote is *self-consistency*,
or best-of-$N$ with a verifier: flat latency, $N$-fold cost, needs checkable answers).

### Consequences for agent design

"Think step by step" adds little when the model already thinks; rigid scaffolds can even
**interfere** with the trained reasoning policy. The prompt shifts from scripting the procedure
to stating **goal, constraints and output contract**. Planning moves inward: plan quality comes
from the budget dial, not a plan template. The agent loop reshapes into **fewer, heavier
iterations** — a deep-plan node up front (reasoning model, large budget), cheap execute–observe
steps in the middle, a verify node at the end, replanning only when verification fails. CoT
prompting still earns its keep: on standard models, for audit trails, and for domain procedures.

### Economics

The lecture's arithmetic: 20 steps × 400 visible output tokens = 8,000 tokens on a standard
model. Add 3,000 thinking tokens per step: $20 \times 3400 = 68{,}000$ billed tokens — $8.5\times$
the volume. With a $4\times$ per-token premium, output cost grows $8.5 \times 4 = 34$-fold for
byte-identical visible text. Latency: 3,000 extra *sequential* tokens at 80 tok/s add
$\approx 38$ s per step. **Rule of thumb: pay for thinking where errors compound or are
expensive to undo — never for formatting.** Hence the hybrid patterns: **model routing** (cheap
by default), **plan–execute split**, **verifier at the end**, **escalation ladder**.

### This lab

You will run both model classes locally via Ollama, read and parse thinking traces, measure
accuracy vs token cost vs latency on a small checkable task suite, plot **accuracy against a
token budget** and find the knee of the curve — exactly what the lecture's lab link announced —
and extend the lecture's minimal router with a **per-run thinking-budget cap**.

## Part A — Setup & Ollama connectivity

This lab needs **two local models**:

| Role in the lab | Lecture name | Default local stand-in | Pull with |
|---|---|---|---|
| Standard model | `standard-mini` | `qwen2.5:7b` | `ollama pull qwen2.5:7b` |
| Reasoning model | `reasoning-xl` | `deepseek-r1:7b` | `ollama pull deepseek-r1:7b` |

Both are configurable via environment variables (`OLLAMA_STANDARD_MODEL`,
`OLLAMA_REASONING_MODEL`). Any local reasoning model that exposes its trace in
`<think>…</think>` tags works — e.g. `deepseek-r1:7b` or `qwen3:8b`. Every LLM cell below
degrades gracefully if a model is missing, so you can work through the notebook with whatever
is pulled.

In [ ]:
import os
import re
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STANDARD_MODEL  = os.environ.get("OLLAMA_STANDARD_MODEL",  "qwen2.5:7b")     # 'standard-mini'
REASONING_MODEL = os.environ.get("OLLAMA_REASONING_MODEL", "deepseek-r1:7b")  # 'reasoning-xl'

OLLAMA_OK = False
AVAILABLE = []


def _field(obj, name, default=None):
    """Read a field from dict-like or attribute-style Ollama responses (client versions differ)."""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


try:
    import ollama
    listing = ollama.list()
    for m in _field(listing, "models", []) or []:
        name = _field(m, "model", None) or _field(m, "name", "")
        if name:
            AVAILABLE.append(name)
    OLLAMA_OK = True
    print("Ollama is running. Local models:", ", ".join(AVAILABLE) or "(none)")
except Exception as exc:
    print("Could not reach Ollama:", exc)
    print("→ Start Ollama with `ollama serve`, then pull the models with")
    print("  `ollama pull qwen2.5:7b` and `ollama pull deepseek-r1:7b`.")


def model_available(tag):
    base = tag.split(":")[0]
    return any(a == tag or a.split(":")[0] == base for a in AVAILABLE)


HAS_STANDARD  = OLLAMA_OK and model_available(STANDARD_MODEL)
HAS_REASONING = OLLAMA_OK and model_available(REASONING_MODEL)

for label, tag, ok in [("standard", STANDARD_MODEL, HAS_STANDARD),
                       ("reasoning", REASONING_MODEL, HAS_REASONING)]:
    status = "OK" if ok else f"MISSING — run `ollama pull {tag}` (this lab degrades gracefully)"
    print(f"{label:>9} model {tag!r}: {status}")

> **Q:** Define a *reasoning model*. What property distinguishes it from a standard instruction-tuned LLM?
<details><summary>Click for answer</summary>

A reasoning model is an LLM trained — typically with reinforcement learning on verifiable
outcomes — to generate an extended internal chain of thought (thinking tokens) before producing
its visible answer. The distinguishing property: deliberation is a <em>learned, default
behaviour</em> encoded in the weights and controllable via a budget parameter — not a behaviour
elicited by prompt instructions, as with chain-of-thought prompting on a standard model.
</details>

## Part B — Task suite & measurement helpers

The build provides a small synthetic suite in `data/tasks.json`: **5 deep tasks** (multi-step
arithmetic, dependency chains, transitive ordering — terrain where thinking pays) and
**3 shallow tasks** (extraction, formatting — where the lecture's comparison table predicts
*no gain* and wasted thinking tokens). Every task has a checkable answer, so accuracy is
mechanically measurable — the same *verifiable outcome* property that made RL training of
reasoning models possible in the first place.

We ask every model to end its reply with `ANSWER: <answer>` so grading stays mechanical.

In [ ]:
DATA_PATH = "data/tasks.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(___)          # parse the opened JSON file object

tasks_df = pd.DataFrame(___)        # tabular view of the task list
print(f"{len(tasks)} tasks: {(tasks_df['kind'] == 'deep').sum()} deep, "
      f"{(tasks_df['kind'] == 'shallow').sum()} shallow")


def make_prompt(task):
    return (task["question"]
            + "\n\nEnd your reply with a single final line of the form: ANSWER: <your answer>")


tasks_df[["id", "kind", "question", "expected"]]

<details>
<summary><b>Click here for the solution</b></summary>

```python
with open(DATA_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

tasks_df = pd.DataFrame(tasks)
```

</details>

In [ ]:
THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)


def chat_with_stats(model, prompt, num_predict=2048, temperature=0.2):
    """One ollama.chat call with measurements attached."""
    t0 = time.time()
    resp = ollama.chat(
        model=___,                                   # which model to call
        messages=[{"role": "user", "content": prompt}],
        options={"num_predict": num_predict, "temperature": temperature},
    )
    latency = ___ - t0                               # wall-clock seconds for the call
    msg = _field(resp, "message", {})
    content = _field(msg, "content", "") or ""
    thinking = _field(msg, "thinking", None) or ""   # some clients expose the trace separately

    m = THINK_RE.search(___)                         # find an inline <think>…</think> block
    if m:
        thinking = m.group(1).strip()
        content = THINK_RE.sub("", content).strip()

    total_tokens = _field(resp, "eval_count", 0) or 0
    thinking_tokens = int(len(thinking.split()) * 1.3)   # rough token estimate from words

    return {"model": model, "answer": content.strip(), "thinking": thinking,
            "total_tokens": total_tokens, "thinking_tokens": thinking_tokens,
            "latency_s": round(latency, 2)}


def extract_answer(text):
    m = re.search(r"ANSWER\s*:\s*(.+)", text, re.IGNORECASE)
    return (m.group(1) if m else text).strip()


def normalise(s):
    return re.sub(r"[^a-z0-9 ]+", " ", s.lower()).strip()


def is_correct(answer_text, expected):
    ans = normalise(extract_answer(answer_text))
    exp = normalise(str(___))                        # the task's expected answer
    return re.search(rf"\b{re.escape(exp)}\b", ans) is not None


# quick self-test of the grader (no LLM needed)
assert is_correct("blah blah\nANSWER: 34", "34")
assert not is_correct("ANSWER: 340", "34")
assert is_correct("ANSWER: Agentic AI!", "AGENTIC AI")
print("Grader self-test passed.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    resp = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"num_predict": num_predict, "temperature": temperature},
    )
    latency = time.time() - t0
    ...
    m = THINK_RE.search(content)
    ...
    exp = normalise(str(expected))
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `options={"num_predict": ..., "temperature": ...}` — `num_predict` is a **hard cap** on
  generated tokens. We will abuse it in Part E as a crude local thinking-budget dial.
- Latency is wall-clock time around the call, so it includes model loading on the first call —
  run a warm-up call before timing seriously.
- Local reasoning models (`deepseek-r1`, `qwen3`) inline their private chain of thought between
  `<think>` and `</think>` in the message content; some client/server versions expose it in a
  separate `message.thinking` field instead. The helper handles both.
- `eval_count` is the number of **generated (billed) tokens, thinking included** — exactly why
  thinking costs money even when the interface hides it.
- `thinking_tokens ≈ words × 1.3` is a rough heuristic, since Ollama does not report a split
  between thinking and visible tokens.
- The grader parses the `ANSWER:` line, normalises, and matches with a word boundary
  (`\b`) so that `340` does not count as `34`.

</details>

> **Q:** Why are *verifiable rewards* (unit tests, exact answers) central to training reasoning models?
<details><summary>Click for answer</summary>

Reinforcement learning needs a reward signal at scale. Verifiable domains — code that passes
tests, math with a checkable final answer — provide an automatic, cheap and reliable reward
without human judges, enabling millions of training episodes. This is also why early reasoning
models were strongest in math and code, and initially weaker on open-ended tasks where no
automatic verifier exists.
</details>

## Part C — Inside the thinking trace

Local reasoning models like `deepseek-r1` emit their private chain of thought between
`<think>` and `</think>` tags, so — unlike with hosted APIs — you can read the **raw trace**.
The lecture predicts three emergent behaviours: **self-verification**, **backtracking**,
**decomposition**. We run one deep task and count crude lexical markers for each.

Keep the **faithfulness caveat** in mind while reading: the trace is a useful *debugging
hypothesis*, not a reliable account of the computation.

In [ ]:
if HAS_REASONING:
    demo_task = next(t for t in tasks if t["id"] == "deep_cost")
    res = chat_with_stats(___, make_prompt(demo_task), num_predict=4096)

    print(f"Correct: {is_correct(res['answer'], demo_task['expected'])} | "
          f"total tokens: {res['total_tokens']} | latency: {res['latency_s']} s\n")
    print("--- THINKING TRACE (first 1500 chars) ---")
    print(res["thinking"][:1500] or "(model exposed no trace)")
    print("\n--- VISIBLE ANSWER ---")
    print(res["answer"][:500])

    MARKERS = {
        "self-verification": ["check", "verify", "confirm", "double-check"],
        "backtracking":      ["wait", "actually", "hmm", "alternatively", "mistake"],
        "decomposition":     ["first", "then", "next", "step"],
    }
    trace = res["thinking"].lower()
    counts = {behaviour: sum(trace.count(___) for w in words)
              for behaviour, words in MARKERS.items()}
    print("\nMarker counts:", counts)
else:
    print(f"Skipping — reasoning model {REASONING_MODEL!r} not available.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    res = chat_with_stats(REASONING_MODEL, make_prompt(demo_task), num_predict=4096)
    ...
    counts = {behaviour: sum(trace.count(w) for w in words)
              for behaviour, words in MARKERS.items()}
```

Typical trace for `deep_cost`: the model computes 400 + 3000 = 3400 per step, multiplies by 20,
divides by 8000, checks the ratio again ("wait, let me double-check"), then multiplies by 4 —
you should find nonzero counts for all three behaviour groups. The markers are crude lexical
proxies, good enough to see that the behaviours exist; they are not a measurement instrument.

</details>

> **Q:** Explain the *faithfulness caveat* about visible thinking traces.
<details><summary>Click for answer</summary>

Research (Anthropic, 2025) showed that a model's stated reasoning does not reliably reflect the
computation that produced its answer: models sometimes use hints or shortcuts that never appear
in the trace, and the trace can rationalise a conclusion post hoc. Thinking traces are useful
for generating debugging hypotheses, but must not serve as evidence of correctness or safety —
verification has to test <em>outputs</em>, not narratives.
</details>

## Part D — Standard vs reasoning: accuracy, tokens, latency

Now the head-to-head on the full suite — the lab-scale version of the lecture's comparison
table. Expectations from the lecture: the reasoning model wins on **deep** tasks; on
**shallow** tasks it gains nothing and burns thinking tokens ("overthinking trivial steps");
the standard model's typical failure is the **shallow first guess**.

> ⏱️ 2 models × 8 tasks — expect a few minutes, most of it in the reasoning model's calls.

In [ ]:
MODEL_CLASSES = {"standard": STANDARD_MODEL, "reasoning": REASONING_MODEL}
FLAGS = {"standard": HAS_STANDARD, "reasoning": HAS_REASONING}

rows = []
for label, model in MODEL_CLASSES.items():
    if not FLAGS[label]:
        print(f"Skipping {label} — {model!r} not pulled.")
        continue
    for task in tasks:
        res = chat_with_stats(model, make_prompt(task), num_predict=2048)
        correct = is_correct(___, task["expected"])    # grade the visible answer
        rows.append({"model_class": label, "task": task["id"], "kind": task["kind"],
                     "correct": correct, "total_tokens": res["total_tokens"],
                     "thinking_tokens": res["thinking_tokens"],
                     "latency_s": res["latency_s"]})
        print(f"{label:>9} | {task['id']:<15} | correct={correct!s:<5} "
              f"| tokens={res['total_tokens']:>5} | {res['latency_s']:>7.1f} s")

df = pd.DataFrame(rows)
if len(df):
    summary = df.groupby([___, "kind"]).agg(           # aggregate per model class and task kind
        accuracy=("correct", "mean"),
        mean_tokens=("total_tokens", "mean"),
        mean_latency_s=("latency_s", "mean")).round(2)
    display(summary)
else:
    print("No results — pull at least one of the two models to run the comparison.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
        correct = is_correct(res["answer"], task["expected"])
        ...
    summary = df.groupby(["model_class", "kind"]).agg(
        accuracy=("correct", "mean"),
        mean_tokens=("total_tokens", "mean"),
        mean_latency_s=("latency_s", "mean")).round(2)
```

Expected pattern (your exact numbers will differ): on *deep* tasks the reasoning model is more
accurate but several times slower and more token-hungry; on *shallow* tasks both are near 100%
accuracy, and every extra thinking token the reasoning model spends there is pure waste — the
two middle rows of the lecture's table, reproduced on your own hardware.

</details>

In [ ]:
if len(df):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    acc = df.groupby(["model_class", "kind"])[___].mean().unstack()   # accuracy per class/kind
    acc.plot.bar(ax=ax1, rot=0)
    ax1.set_ylabel("accuracy")
    ax1.set_ylim(0, 1.05)
    ax1.set_title("Accuracy by task kind")

    for label, g in df.groupby("model_class"):
        ax2.scatter(g["total_tokens"], g["latency_s"], label=label, alpha=0.7)
    ax2.set_xlabel("billed output tokens (incl. thinking)")
    ax2.set_ylabel(___)                                               # label the y-axis
    ax2.set_title("Token cost vs latency per call")
    ax2.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No results to plot.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
    acc = df.groupby(["model_class", "kind"])["correct"].mean().unstack()
    ...
    ax2.set_ylabel("latency [s]")
```

The right panel makes the economics visible: latency grows with billed tokens because
generation is sequential — the reasoning model's points sit far up and to the right.

</details>

> **Q:** On which task types does increasing the thinking budget yield large gains, and on which almost none? Why?
<details><summary>Click for answer</summary>

Large gains on tasks with deep dependency structure where errors can be detected and corrected
during search: competition math, debugging, multi-step planning. Near-zero gains on knowledge
recall, extraction, classification and formatting: these are shallow — the model either has the
knowledge or it does not, and deliberation cannot manufacture missing facts. This asymmetry is
the economic basis for routing.
</details>

> **📝 Report task R1:** Using your Part D measurements, decide for two concrete research-agent subtasks — (a) the **planner** (decompose a research question into a step plan) and (b) the **formatter** (convert extracted findings into a Markdown table) — which model class each should run on. Justify with your own accuracy, token and latency numbers, and connect your decision to the lecture's rule of thumb ("pay for thinking where errors compound — never for formatting").
> *No solution is provided — include your answer and a short justification in your lab report.*

## Part E — Accuracy vs thinking budget: plot the curve yourself

Exactly what the lecture's lab link announced: run the deep tasks at several **token budgets**
and find the knee of the curve. Local models expose no native effort dial, so we approximate
the budget with Ollama's `num_predict` hard cap: a call that is cut off mid-thought never
reaches its `ANSWER:` line and is graded wrong — a crude but honest local stand-in for a
thinking-budget parameter.

One eval-hygiene detail (claim hygiene, Section 5 of the lecture): a truncated trace might
*accidentally* contain the expected number somewhere in its thinking, so for this experiment we
grade **strictly** — no `ANSWER:` line, no credit.

> ⏱️ 4 budgets × 3 deep tasks on the reasoning model — the slowest cell of the lab.

In [ ]:
def is_correct_strict(answer_text, expected):
    """Only grade replies that actually reached an ANSWER: line (truncated runs score 0)."""
    if not re.search(r"ANSWER\s*:", answer_text, re.IGNORECASE):
        return False
    return is_correct(answer_text, expected)


BUDGETS = [128, 384, 1024, 3072]
deep_tasks = [t for t in tasks if t["kind"] == "deep"][:3]

budget_rows = []
if HAS_REASONING:
    for budget in BUDGETS:
        for task in deep_tasks:
            res = chat_with_stats(REASONING_MODEL, make_prompt(task), num_predict=___)
            full_text = res["answer"] if res["answer"] else res["thinking"]
            budget_rows.append({"budget": budget,
                                "correct": is_correct_strict(full_text, task["expected"]),
                                "tokens": res["total_tokens"]})
        n_ok = sum(r["correct"] for r in budget_rows if r["budget"] == budget)
        print(f"budget {budget:>5}: {n_ok}/{len(deep_tasks)} correct")
else:
    print(f"Skipping — reasoning model {REASONING_MODEL!r} not available.")

if budget_rows:
    curve = pd.DataFrame(budget_rows).groupby("budget").agg(
        accuracy=("correct", ___), mean_tokens=("tokens", "mean"))
    display(curve)

    ax = curve["accuracy"].plot(marker="o")
    ax.set_xscale(___)                 # the lecture's claim is *log*-linear
    ax.set_xlabel("token budget (num_predict)")
    ax.set_ylabel("accuracy on deep tasks")
    ax.set_ylim(0, 1.05)
    ax.set_title("Accuracy vs test-time compute — find the knee")
    plt.show()

<details>
<summary><b>Click here for the solution</b></summary>

```python
            res = chat_with_stats(REASONING_MODEL, make_prompt(task), num_predict=budget)
    ...
    curve = pd.DataFrame(budget_rows).groupby("budget").agg(
        accuracy=("correct", "mean"), mean_tokens=("tokens", "mean"))
    ...
    ax.set_xscale("log")
```

Typical shape: 0% at 128 (cut off mid-thought), a steep rise once the budget covers a complete
trace plus the answer, then a flat tail — going from 1024 to 3072 usually buys little. On the
log x-axis the rising part looks roughly linear: your lab-scale replication of the published
inference-time scaling curves, knee included.

</details>

> **Q:** What does a *roughly log-linear* relation between thinking tokens and accuracy imply for cost planning?
<details><summary>Click for answer</summary>

Each constant increment in accuracy costs a multiplicative increase in tokens: going from 1k to
10k thinking tokens might buy a similar gain as going from 10k to 100k. Marginal accuracy is
therefore increasingly expensive, and there is a task-dependent knee beyond which further
budget mostly buys verbosity. Set the budget near that knee — found empirically, exactly as in
Part E.
</details>

> **📝 Report task R2:** Read the knee off your Part E curve: which budget would you configure for the research agent's planning step in production, and why? Relate your curve to the lecture's two claims — accuracy grows *roughly log-linearly* with thinking tokens, and budgets beyond task difficulty buy *verbosity, not correctness*. Include the plot in your report.
> *No solution is provided — include your answer and the plot in your lab report.*

## Part F — The minimal model router, extended

The lecture's twelve-line router sends `plan` and `verify` steps (and retries) to the reasoning
model and everything else to the cheap one — *model choice is config, not architecture*. We
rebuild it and attach the lecture's cost arithmetic, fed with **your own Part D measurements**
where available (with the lecture's illustrative numbers as fallback). No LLM calls needed
here — this is napkin arithmetic, made executable.

In [ ]:
# Per-call cost estimates: your Part D measurements, lecture numbers as fallback.
def _mean(model_class, col, fallback):
    try:
        v = df[df["model_class"] == model_class][col].mean()
        return float(v) if pd.notna(v) else fallback
    except Exception:
        return fallback


EST = {
    "standard":  {"visible": 400, "thinking": 0,
                  "latency": _mean("standard", "latency_s", 5.0)},
    "reasoning": {"visible": 400,
                  "thinking": _mean("reasoning", "thinking_tokens", 3000.0),
                  "latency": _mean("reasoning", "latency_s", 42.0)},
}
PRICE = {"standard": 1.0, "reasoning": 4.0}      # relative price per token (lecture: 4x premium)

# A simulated 20-step research-agent run: one plan, mechanics in between, one verify.
STEPS = ([{"kind": "plan", "retries": 0}]
         + [{"kind": k, "retries": 0} for k in
            ["search", "extract", "search", "extract", "search", "extract",
             "search", "extract", "draft", "search", "extract", "search",
             "extract", "draft", "draft", "format", "draft", "format"]]
         + [{"kind": "verify", "retries": 0}])
assert len(STEPS) == 20

PLAN_MODEL, WORK_MODEL = "reasoning", "standard"


def model_for(step):
    if step["kind"] in (___, ___):     # the steps where errors compound / quality is decided
        return PLAN_MODEL
    if step["retries"] > ___:          # escalate after a failure
        return PLAN_MODEL
    return WORK_MODEL


def run_cost(policy):
    """policy: step -> model class. Returns billed tokens, cost units, latency in minutes."""
    billed = cost = latency = 0.0
    for step in STEPS:
        m = policy(step)
        tokens = EST[m]["visible"] + EST[m][___]   # thinking is billed like output
        billed += tokens
        cost += tokens * PRICE[m]
        latency += EST[m]["latency"]
    return {"billed_tokens": int(billed), "cost_units": round(cost),
            "latency_min": round(latency / 60, 1)}


policies = {"all-standard": lambda s: "standard",
            "all-reasoning": lambda s: "reasoning",
            "routed": model_for}
pd.DataFrame({name: run_cost(p) for name, p in policies.items()}).T

<details>
<summary><b>Click here for the solution</b></summary>

```python
def model_for(step):
    if step["kind"] in ("plan", "verify"):
        return PLAN_MODEL
    if step["retries"] > 0:
        return PLAN_MODEL
    return WORK_MODEL

def run_cost(policy):
    ...
        tokens = EST[m]["visible"] + EST[m]["thinking"]
```

With the lecture's fallback numbers you reproduce its arithmetic: all-reasoning bills 68,000
tokens (8.5× the volume) at ~34× the cost of all-standard, while the routed run — two expensive
calls bracketing eighteen cheap ones, the *barbell* cost shape — collapses most of that
multiplier. With your own measured 7B numbers the multipliers are smaller, but the shape is
the same.

</details>

> **Q:** Short check — in the lecture's minimal router, which two conditions route a step to the reasoning model?
<details><summary>Click for answer</summary>

Step kind being <code>plan</code> or <code>verify</code>, and the step having a retry count
above zero — i.e. the cheap model already failed at it once, so the retry escalates to the
stronger model. Everything else defaults to the cheap model.
</details>

> **📝 Report task R3 (code):** Complete the cell below — extend the router with a **per-run thinking-budget cap**, exactly what the lecture announced for this lab. Once the run's cumulative thinking tokens reach `THINKING_CAP`, every remaining step must run on the cheap model, whatever its kind. Decide — and defend in your report — what your cap means for a `verify` step that arrives after the budget is exhausted.
> *No solution is provided — include your code and a short justification in your lab report.*

In [ ]:
# 📝 Report task R3 — no solution fold-out for this cell.
THINKING_CAP = 8000     # max thinking tokens one run may spend


def model_for_capped(step, spent_thinking):
    # Route like model_for, but enforce the per-run cap:
    # once spent_thinking >= THINKING_CAP, everything runs on WORK_MODEL.
    ___


spent, log_rows = 0, []
for step in STEPS:
    m = model_for_capped(step, spent)
    if m == PLAN_MODEL:
        spent += EST["reasoning"]["thinking"]
    log_rows.append({"step": step["kind"], "model": m, "thinking_spent": int(spent)})

pd.DataFrame(log_rows)

> **📝 Report task R4:** Redo the lecture's thinking-token arithmetic with **measured** numbers: from your Part D data, take the reasoning model's mean thinking tokens per call and mean latency, and recompute for the 20-step run (i) the billed-token multiplier vs all-standard, (ii) the cost multiplier at a 4× price premium, and (iii) the added wall-clock time. Compare with the lecture's illustrative 34× / ~13 min and explain any difference in one short paragraph.
> *No solution is provided — show the arithmetic in your lab report.*

> **Q (not exam-relevant):** By mid-2026, *hybrid* reasoning designs dominate. What does hybrid mean here, and why did it win over separate reasoning models?
<details><summary>Click for answer</summary>

Hybrid means a single model that can answer immediately or think at length, with thinking depth
set per request via a budget or effort parameter. It won because agent workloads mix shallow and
deep steps: a per-request dial lets one deployment serve both, avoids maintaining two model
integrations, and turns reasoning depth into a routine engineering decision made in code, per
call.
</details>

## Part G — Tuning & exploration (no gaps)

Things to play with — none of these cells contain gaps:

- **Parallel test-time compute:** the cell below implements **self-consistency** (Wang et al.,
  2023) — sample $N$ chains at nonzero temperature and majority-vote the answer. This gives the
  *standard* model the parallel dial that the reasoning model has serially. Vary $N$ and the
  temperature: when does the vote flip a wrong single answer into a correct majority?
- **Budgets:** change `BUDGETS` in Part E — can you locate the knee more precisely with
  intermediate budgets?
- **Models:** set `OLLAMA_REASONING_MODEL=qwen3:8b` (a hybrid thinker) and rerun Parts C–E. Do
  the trace markers and the knee change?
- **Temperature for reasoning models:** `deepseek-r1` is documented to work best around 0.6 —
  rerun Part D with `temperature=0.6` and compare accuracy and trace length.
- **Router policies:** in Part F, add retries to some `STEPS` and watch the escalation rule
  shift cost from the cheap to the expensive column.

In [ ]:
from collections import Counter


def self_consistency(model, task, n=5, temperature=0.8, num_predict=768):
    """Parallel test-time compute: sample n answers, return the majority vote."""
    votes, total_tokens = [], 0
    for _ in range(n):
        res = chat_with_stats(model, make_prompt(task), num_predict=num_predict,
                              temperature=temperature)
        votes.append(normalise(extract_answer(res["answer"])))
        total_tokens += res["total_tokens"]
    winner, n_votes = Counter(votes).most_common(1)[0]
    return {"votes": votes, "majority": winner, "n_votes": n_votes,
            "correct": is_correct("ANSWER: " + winner, task["expected"]),
            "total_tokens": total_tokens}


sc_task = next(t for t in tasks if t["id"] == "deep_chain")

if HAS_STANDARD:
    out = self_consistency(STANDARD_MODEL, sc_task, n=5, temperature=0.8)
    print(f"Votes:    {out['votes']}")
    print(f"Majority: {out['majority']!r} ({out['n_votes']}/5) — correct: {out['correct']}")
    print(f"Total tokens across samples: {out['total_tokens']} "
          f"(N-fold cost; latency stays flat if the samples run in parallel)")
else:
    print(f"Skipping — standard model {STANDARD_MODEL!r} not available.")

# Optional interactive exploration (falls back gracefully without ipywidgets)
try:
    from ipywidgets import interact, IntSlider, FloatSlider

    def explore(n=5, temperature=0.8):
        if HAS_STANDARD:
            out = self_consistency(STANDARD_MODEL, sc_task, n=n, temperature=temperature)
            print(out["votes"], "→", out["majority"], "| correct:", out["correct"],
                  "| tokens:", out["total_tokens"])
        else:
            print("standard model missing")

    interact(explore, n=IntSlider(5, 1, 9, 2), temperature=FloatSlider(0.8, 0.0, 1.2, 0.1))
except Exception:
    print("ipywidgets not available — call self_consistency(...) with different "
          "n / temperature by hand.")

## Wrap-up

**Takeaways**

- A reasoning model's deliberation is **learned via RL on verifiable outcomes** — you dial *how
  much* it thinks; you no longer script *how*.
- **Test-time compute is the third scaling axis**: your Part E curve is the lab-scale version of
  the published log-linear plots — and it has a knee.
- The gains are **task-dependent**: deep tasks profit, shallow tasks pay pure overhead — your
  Part D table shows both, on your own hardware.
- **Traces are hypotheses, not evidence** — verification must test outputs, not narratives.
- **Routing turns model choice into configuration**: plan and verify get the expensive model,
  retries escalate, and a per-run thinking-budget cap bounds the worst case.

**Next week (Session 06 — Orchestration):** who plans, who executes, who verifies — ReAct,
plan-and-execute, DAG-based vs dynamic control. Today's vocabulary (heavy vs light steps, plan
vs execute) is exactly what those patterns arrange.

**📝 For your lab report — checklist**

| # | Task | Where |
|---|------|-------|
| R1 | Model class per subtask (planner vs formatter), justified with your measurements | Part D |
| R2 | Knee of the accuracy-vs-budget curve + production budget choice (incl. plot) | Part E |
| R3 | Code: router extended with a per-run thinking-budget cap + justification | Part F |
| R4 | Lecture cost arithmetic redone with your measured numbers | Part F |